In [1]:
import sys
import os
import torch
from dotenv import load_dotenv
# Add the parent directory to the system path for module imports
sys.path.append('../')

import LLMP as L

# Clear CUDA cache if using GPU
torch.cuda.empty_cache()

# Load environment variables from the .env file
load_dotenv()

# Create instances of the GPT and Gemini models
gpt4vision = L.GPTModel("gpt-4-turbo")
gpt4o = L.GPTModel("gpt-4o")
gemini1 = L.GeminiProVision()  
gemini2 = L.Gemini1_5Flash() 

/home/huuthanhvy.nguyen001/anaconda3/envs/LLMP/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2024-09-16 07:47:41,165] [INFO] [real_accelerator.py:191:get_accelerator] Setting ds_accelerator to cuda (auto detect)
Initializing GPTModel with model_name: gpt-4-turbo
Initializing GPTModel with model_name: gpt-4o


# Chain of Thought

In [14]:
query = """
Observe the two lines depicted in the image. 
Begin by determining the point where the lines intersect. 
Next, think about how these lines create an angle at this point. 
Calculate the degree of this angle. Finally, your answer should be a number only, representing the angle, with no additional text or characters.
Let's think step by step.
"""

images = [L.GPImage.figure1('angle') for i in range(1)]

# Run the evaluator
result = L.Evaluator.run(images, query, all_model_instances)
print(result)

/home/huuthanhvy.nguyen001/anaconda3/envs/LLMP/lib/python3.11/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
Loading checkpoint shards: 100%|██████████████████| 2/2 [00:05<00:00,  2.78s/it]
/home/huuthanhvy.nguyen001/anaconda3/envs/LLMP/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:389: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/huuthanhvy.nguyen001/anaconda3/envs/LLMP/lib/python3.11/site-packages/transformers/generation/

{'gt': [62], 'gpt4o': {'raw_answers': [' 90'], 'parsed_answers': [[90.0]], 'mean': 90.0, 'std': 0.0, 'mse': 784.0, 'mlae': 11.451275516423348, 'times': [5895.834445953369], 'forced_repetitions': 0}, 'gpt4vision': {'raw_answers': ['45'], 'parsed_answers': [[45.0]], 'mean': 45.0, 'std': 0.0, 'mse': 289.0, 'mlae': 10.731425107642949, 'times': [6201.482772827148], 'forced_repetitions': 0}, 'LLaVA': {'raw_answers': ['The two lines intersect at a point, and they create an angle of 45 degrees.'], 'parsed_answers': [[45.0]], 'mean': 45.0, 'std': 0.0, 'mse': 289.0, 'mlae': 10.731425107642949, 'times': [11106.823921203613], 'forced_repetitions': 0}, 'CustomLLaVA': {'raw_answers': ['5, 0.875,'], 'parsed_answers': [[5.0, 0.875]], 'mean': 2.9375, 'std': 0.0, 'mse': 3488.37890625, 'mlae': 12.52805724178844, 'times': [9970.689535140991], 'forced_repetitions': 0}, 'GeminiProVision': {'raw_answers': ['45 \n'], 'parsed_answers': [[45.0]], 'mean': 45.0, 'std': 0.0, 'mse': 289.0, 'mlae': 10.73142510764294

In [22]:
import pandas as pd

# Extracting data into a structured format
rows = []

for model, metrics in result.items():
    if model != 'gt':  # Ignore 'gt'
        rows.append({
            'Model': model,
            'Mean': metrics['mean'],
            'MSE': metrics['mse'],
            'MLAE': metrics['mlae'],
            'Time (ms)': metrics['times'][0]
        })

# Create DataFrame
df = pd.DataFrame(rows)

# Round the values
df = df.round({'Mean': 2, 'MSE': 2, 'MLAE': 2, 'Time (ms)': 2})

# Sort by MSE and MLAE
df = df.sort_values(by=['MSE', 'MLAE'])

# Apply custom coloring based on ranking
def highlight_top_and_last(row):
    if row.name == df.index[0]:  # Top 1
        return ['background-color: yellow'] * len(row)
    elif row.name == df.index[1]:  # Top 2
        return ['background-color: yellow'] * len(row)
    elif row.name == df.index[-1]:  # Last (highest MSE and MLAE)
        return ['background-color: lightgreen'] * len(row)
    return [''] * len(row)

# Apply the highlighting function
df_styled = df.style.apply(highlight_top_and_last, axis=1)

# Display the styled DataFrame
df_styled

,Model,Mean,MSE,MLAE,Time (ms)
1,gpt4vision,45.000000,289.000000,10.730000,6201.480000
2,LLaVA,45.000000,289.000000,10.730000,11106.820000
4,GeminiProVision,45.000000,289.000000,10.730000,5752.130000
0,gpt4o,90.000000,784.000000,11.450000,5895.830000
5,Gemini1_5Flash,90.000000,784.000000,11.450000,5348.600000
3,CustomLLaVA,2.940000,3488.380000,12.530000,9970.690000


# Contextual Prompting

### ChatGPT

In [10]:
query = """The image shows two lines crossing each other, creating an acute angle. 
Acute angles are less than 90 degrees and are found in shapes like triangles. 
Estimate the angle's range within 10 degrees and provide only the numerical value without any extra text or symbols."""

images = [L.GPImage.figure1('angle') for i in range(1)]

# Run the evaluator
result = L.Evaluator.run(images, query, chatgpt)
print(result)

{'gt': [35], 'gpt4o': {'raw_answers': ['50'], 'parsed_answers': [[50.0]], 'mean': 50.0, 'std': 0.0, 'mse': 225.0, 'mlae': 10.550867004960905, 'times': [5963.450908660889], 'forced_repetitions': 0}, 'gpt4vision': {'raw_answers': ['40-50'], 'parsed_answers': [[40.0, 50.0]], 'mean': 45.0, 'std': 0.0, 'mse': 100.0, 'mlae': 9.965964610272083, 'times': [6425.282955169678], 'forced_repetitions': 0}}


### Gemini

In [11]:
# Run the evaluator
result = L.Evaluator.run(images, query, gemini)
print(result)

{'gt': [35], 'GeminiProVision': {'raw_answers': ['50 \n'], 'parsed_answers': [[50.0]], 'mean': 50.0, 'std': 0.0, 'mse': 225.0, 'mlae': 10.550867004960905, 'times': [5783.552646636963], 'forced_repetitions': 0}, 'Gemini1_5Flash': {'raw_answers': ['40-50 \n'], 'parsed_answers': [[40.0, 50.0]], 'mean': 45.0, 'std': 0.0, 'mse': 100.0, 'mlae': 9.965964610272083, 'times': [5341.695070266724], 'forced_repetitions': 0}}


### LLaVA

In [12]:
# Run the evaluator
result = L.Evaluator.run(images, query, llava)
print(result)

/home/huuthanhvy.nguyen001/anaconda3/envs/LLMP/lib/python3.11/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
Loading checkpoint shards: 100%|██████████████████| 2/2 [00:05<00:00,  2.86s/it]
/home/huuthanhvy.nguyen001/anaconda3/envs/LLMP/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:389: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/huuthanhvy.nguyen001/anaconda3/envs/LLMP/lib/python3.11/site-packages/transformers/generation/

KeyboardInterrupt: 

# Three of Thought

In [13]:
query = """
Consider possible acute angles like 30°, 45°, or 60°. 
Reflect on why each might match the angle in the image. 
Choose the most likely angle, ensuring it’s within 10 degrees. 
Provide only the number, without any additional text or symbols."""

images = [L.GPImage.figure1('angle') for i in range(1)]

# Run the evaluator
result = L.Evaluator.run(images, query, chatgpt)
print(result)

{'gt': [63], 'gpt4o': {'raw_answers': ['45'], 'parsed_answers': [[45.0]], 'mean': 45.0, 'std': 0.0, 'mse': 324.0, 'mlae': 10.813881374894095, 'times': [6445.10817527771], 'forced_repetitions': 0}, 'gpt4vision': {'raw_answers': ['45'], 'parsed_answers': [[45.0]], 'mean': 45.0, 'std': 0.0, 'mse': 324.0, 'mlae': 10.813881374894095, 'times': [6115.251541137695], 'forced_repetitions': 0}}


In [14]:
# Run the evaluator
result = L.Evaluator.run(images, query, gemini)
print(result)

{'gt': [63], 'GeminiProVision': {'raw_answers': ['45 \n'], 'parsed_answers': [[45.0]], 'mean': 45.0, 'std': 0.0, 'mse': 324.0, 'mlae': 10.813881374894095, 'times': [5678.170204162598], 'forced_repetitions': 0}, 'Gemini1_5Flash': {'raw_answers': ['45\n'], 'parsed_answers': [[45.0]], 'mean': 45.0, 'std': 0.0, 'mse': 324.0, 'mlae': 10.813881374894095, 'times': [5320.955514907837], 'forced_repetitions': 0}}


In [ ]:
# Run the evaluator
result = L.Evaluator.run(images, query, llava)
print(result)

/home/huuthanhvy.nguyen001/anaconda3/envs/LLMP/lib/python3.11/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
Loading checkpoint shards: 100%|██████████████████| 2/2 [00:05<00:00,  2.85s/it]
/home/huuthanhvy.nguyen001/anaconda3/envs/LLMP/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:389: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/huuthanhvy.nguyen001/anaconda3/envs/LLMP/lib/python3.11/site-packages/transformers/generation/